In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
# EMP05 debug controls
# Options: main_only, contingent_all, all_workers
dbutils.widgets.dropdown("emp05_debug_scope", "main_only", ["main_only", "contingent_all", "all_workers"])
print("emp05_debug_scope =", dbutils.widgets.get("emp05_debug_scope"))

In [ ]:
%sql
/* ===================================================================================
   METRIC: EMP05 - Contingent Workers in High-Risk Jurisdictions
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
   2. HIGH-RISK JURISDICTIONS: Queries the td_country_risk_rating_abac table to isolate 
      countries where the FinalABACRating is strictly 'High'.
   3. CONTINGENT WORKERS: Filters the HR List for 'contingent' workers. Applies the 
      "Smart Padding" rule (pads leading zeros ONLY if the ID is pure numbers and 
      strictly under 4 digits) to perfectly match the CC Bootstrap as a string.
   4. FILTERING & MAPPING: INNER JOINS the contingent workers to the High-Risk table. 
      Then maps them to AU_IDs via the CC Bootstrap.
   5. OUTPUT FORMATTING: Rolls up the data by AU_ID. If mapped high-risk workers 
      exist, outputs 'Yes'. Otherwise, 'No'.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE UPPER(TRIM(FinalABACRating)) = 'HIGH'
),

Contingent_Workers AS (
    -- Apply the Smart Padding rule to HR data to prevent alphanumeric dropping
    SELECT 
        TRIM(CostCenterID) AS Raw_CC,
        CASE 
            WHEN TRIM(CostCenterID) RLIKE '^[0-9]+$' AND LENGTH(TRIM(CostCenterID)) < 4 
            THEN LPAD(TRIM(CostCenterID), 4, '0')
            ELSE TRIM(CostCenterID) 
        END AS Cleaned_CC,
        UPPER(TRIM(Workcountry)) AS Workcountry,
        TRIM(WorkerType) AS WorkerType
    FROM hive_metastore.ra_adido_2025.hrbirai_17930_enterprise_list_as_of_103125_new_employees_101025_10312025
    WHERE LOWER(TRIM(WorkerType)) LIKE '%contingent%'
),

High_Risk_Contingent_Workers AS (
    SELECT 
        c.Cleaned_CC,
        c.Workcountry
    FROM Contingent_Workers c
    INNER JOIN High_Risk_Countries h
        ON c.Workcountry = h.CountryName
    WHERE c.Cleaned_CC IS NOT NULL AND c.Cleaned_CC != ''
),

CC_Mapping AS (
    -- Pull directly from the bootstrap view (no INT casting)
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        TRIM(CAST(Cost_Center_ID AS STRING)) AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND Cost_Center_ID IS NOT NULL
),

Flagged_AUs AS (
    SELECT 
        m.AU_ID,
        COUNT(*) AS Flagged_Count
    FROM High_Risk_Contingent_Workers hw
    INNER JOIN CC_Mapping m
        ON hw.Cleaned_CC = m.Mapped_CC
    GROUP BY m.AU_ID
)

SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'EMP05' AS QuestionID,               
    CASE 
        WHEN COALESCE(f.Flagged_Count, 0) > 0 THEN 'Yes' 
        ELSE 'No' 
    END AS Response,
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Flagged_AUs f
    ON a.BusinessID = f.AU_ID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EMP05 Drill-Down (Pipeline Tracker)
   
   PURPOSE: Shows EVERY contingent worker found in the HR data. Uses LEFT JOINs to 
   expose exactly where a record fails (e.g., Country spelling mismatch, unmapped CC, 
   or Master AU drop).
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName, 'HIGH' AS Risk_Level
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE UPPER(TRIM(FinalABACRating)) = 'HIGH'
),

Contingent_Workers AS (
    SELECT 
        TRIM(CostCenterID) AS Raw_CC,
        CASE 
            WHEN TRIM(CostCenterID) RLIKE '^[0-9]+$' AND LENGTH(TRIM(CostCenterID)) < 4 
            THEN LPAD(TRIM(CostCenterID), 4, '0')
            ELSE TRIM(CostCenterID) 
        END AS Cleaned_CC,
        UPPER(TRIM(Workcountry)) AS Workcountry,
        TRIM(WorkerType) AS WorkerType
    FROM hive_metastore.ra_adido_2025.hrbirai_17930_enterprise_list_as_of_103125_new_employees_101025_10312025
    WHERE LOWER(TRIM(WorkerType)) LIKE '%contingent%'
),

CC_Mapping AS (
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        TRIM(CAST(Cost_Center_ID AS STRING)) AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND Cost_Center_ID IS NOT NULL
)

SELECT 
    c.Raw_CC,
    c.Cleaned_CC AS Smart_Padded_CC,
    c.WorkerType,
    c.Workcountry AS HR_Country_Spelling,
    h.CountryName AS Matched_Risk_Country,
    m.AU_ID AS Bridged_AU_ID,
    
    CASE 
        WHEN h.CountryName IS NULL THEN '❌ DROPPED: HR Country is not in High-Risk List'
        WHEN m.AU_ID IS NULL THEN '❌ DROPPED: Cost Center failed to map to AU'
        WHEN mast.BusinessID IS NULL THEN '❌ DROPPED: Mapped AU is not in Master Canada/HK/BB List'
        ELSE '✅ KEPT: High-Risk Contingent mapped successfully'
    END AS Pipeline_Status

FROM Contingent_Workers c
-- LEFT JOINS expose exactly where the chain breaks
LEFT JOIN High_Risk_Countries h ON c.Workcountry = h.CountryName
LEFT JOIN CC_Mapping m ON c.Cleaned_CC = m.Mapped_CC
LEFT JOIN Master_AUs mast ON m.AU_ID = mast.BusinessID
ORDER BY Pipeline_Status DESC, c.Cleaned_CC;